# `regionprops` benchmark — scikit-image side

Times `skimage.measure.regionprops_table` on the shared fixture with `timeit`.

```
cd benchmarks/python && uv run jupyter lab ../notebooks/03-python-timeit.ipynb
```

In [1]:
import hashlib
import json
import platform
import timeit
from pathlib import Path

import numpy as np
import pandas as pd
import skimage
from skimage.measure import regionprops_table

BENCH = Path.cwd()
while not (BENCH / "fixtures").is_dir() and BENCH != BENCH.parent:
    BENCH = BENCH.parent
FIXTURES, RESULTS = BENCH / "fixtures", BENCH / "results"

PROPERTY_MAP = {
    "area": "area",
    "perimeter": "perimeter",
    "convex_area": "area_convex",
    "solidity": "solidity",
    "major_axis_length": "axis_major_length",
    "minor_axis_length": "axis_minor_length",
}
print("scikit-image", skimage.__version__)

scikit-image 0.26.0


## Fixture

The SHA-256 is asserted so both languages are measuring the same array.

In [2]:
def load_fixture(name):
    binpath, metapath = FIXTURES / f"labels_{name}.bin", FIXTURES / f"labels_{name}.json"
    meta = json.loads(metapath.read_text())
    assert hashlib.sha256(binpath.read_bytes()).hexdigest() == meta["sha256"], \
        f"{name}: fixture digest mismatch -- regenerate with export_labels.jl"
    labels = np.fromfile(binpath, dtype="<i4").reshape(meta["rows"], meta["cols"])
    return labels, meta

labels, meta = load_fixture("full")
print(f"{meta['name']}: {meta['rows']} x {meta['cols']} "
      f"({labels.size/1e6:.1f} Mpx), {meta['n_labels']} labels")

full: 5680 x 3392 (19.3 Mpx), 1910 labels


## What is timed

`regionprops_table` (Julia) keeps labels with `area > 1` — *strictly* greater — so
single-pixel regions are dropped; scikit-image keeps them. The filter is inside
the timed function because the Julia side pays for its equivalent too.

In [3]:
def call(labels, properties):
    props = tuple(dict.fromkeys(("label", "area") + tuple(properties)))
    table = regionprops_table(labels, properties=props)
    keep = table["area"] > 1
    return {k: v[keep] for k, v in table.items()}

_t = call(labels, tuple(PROPERTY_MAP.values()))
print("rows:", len(_t["label"]))

rows: 1834


In [4]:
def bench(fn, *, repeat=5):
    """
    Minimum-of-`repeat` wall time for `fn()`, in seconds.

    `setup` re-enables the GC inside the timed callable, matching the Julia
    side, which runs with it on. `number=1` because one call costs
    milliseconds or more.
    """
    fn()        # warmup
    times = timeit.Timer(
        "fn()", setup="import gc; gc.enable()", globals={"fn": fn}
    ).repeat(repeat=repeat, number=1)
    return {"min_s": min(times), "median_s": float(np.median(times)),
            "samples": len(times)}

## Timings

Each property alone, then all six in one call. The minimum is reported, as on
the Julia side.

In [5]:
REPEAT = 5

rows = [{"case": "single", "property": jname,
         **bench(lambda sk=skname: call(labels, (sk,)), repeat=REPEAT)}
        for jname, skname in PROPERTY_MAP.items()]
rows.append({"case": "combined", "property": "all_properties",
             **bench(lambda: call(labels, tuple(PROPERTY_MAP.values())), repeat=REPEAT)})

timings = pd.DataFrame(rows)
timings["min_ms"] = timings["min_s"] * 1e3
timings[["case", "property", "min_ms", "median_s", "samples"]]

,case,property,min_ms,median_s,samples
0,single,area,60.889333,0.062498,5
1,single,perimeter,148.215706,0.149662,5
2,single,convex_area,545.395840,0.549160,5
3,single,solidity,548.302527,0.594262,5
4,single,major_axis_length,292.208299,0.292814,5
5,single,minor_axis_length,292.103588,0.296523,5
6,combined,all_properties,870.856065,0.895931,5


In [6]:
# The isolated calls do not sum to the combined one: work is shared within a call.
singles = timings.loc[timings["case"] == "single", "min_s"].sum()
combined = timings.loc[timings["case"] == "combined", "min_s"].iloc[0]
print(f"sum of isolated calls : {singles:.4f} s")
print(f"one combined call     : {combined:.4f} s  "
      f"({100*(singles-combined)/singles:.1f}% saved)")

sum of isolated calls : 1.8871 s
one combined call     : 0.8709 s  (53.9% saved)


## Against the Julia side

Run `benchmark_regionprops.jl` first to populate this.

In [7]:
jpath = RESULTS / "julia.json"
if not jpath.exists():
    print("Run benchmark_regionprops.jl to populate this section.")
    comparison = None
else:
    jrows = {(r["case"], r["property"]): r
             for r in json.loads(jpath.read_text())["results"]}
    comparison = pd.DataFrame([
        {"property": t["property"],
         "julia_ms": jrows[k]["min_s"] * 1e3,
         "skimage_ms": t["min_s"] * 1e3,
         "ratio_py_over_jl": t["min_s"] / jrows[k]["min_s"]}
        for _, t in timings.iterrows()
        if (k := (t["case"], t["property"])) in jrows
    ])
comparison

,property,julia_ms,skimage_ms,ratio_py_over_jl
0,area,122.478613,60.889333,0.497143
1,perimeter,2474.817433,148.215706,0.059890
2,convex_area,2466.181231,545.395840,0.221150
3,solidity,2415.975493,548.302527,0.226949
4,major_axis_length,469.698171,292.208299,0.622119
5,minor_axis_length,379.530759,292.103588,0.769644
6,all_properties,3883.426597,870.856065,0.224249


## Export

Same shape as `julia.json`, read by `join_results.jl`.

In [8]:
SAVE = False        # flip to True to write the file

payload = {
    "env": {
        "python": platform.python_version(),
        "scikit_image": skimage.__version__,
        "numpy": np.__version__,
        "estimator": "minimum",
        "timer": "timeit (gc enabled, number=1)",
        "machine": platform.platform(),
    },
    "results": [
        {"case": r["case"], "impl": "python", "property": r["property"],
         "size": meta["name"], "n_px": int(meta["rows"] * meta["cols"]),
         "n_labels": int(meta["n_labels"]), "min_s": float(r["min_s"]),
         "median_s": float(r["median_s"]), "samples": int(r["samples"])}
        for _, r in timings.iterrows()
    ],
}

out = RESULTS / "python_timeit.json"
if SAVE:
    RESULTS.mkdir(parents=True, exist_ok=True)
    out.write_text(json.dumps(payload, indent=1))
    print(f"wrote {out} ({len(payload['results'])} rows)")
else:
    print("SAVE is False -- set it to True to write", out)

SAVE is False -- set it to True to write /home/cpaniaguam/IceFloeTracker.jl/benchmarks/results/python_timeit.json


Value parity is in `02-comparison.ipynb`; it does not depend on the timing.